# Data Ingestion & Exploration

Objetivo: cargar el dataset raw de HM Land Registry (`price_paid`), inspeccionar su esquema y estadísticas básicas, y persistir la capa Bronze en Delta Lake.

Paso 1: Carga y corrección de tipos   
Paso 2: EDA básico (precios, propiedades, años)   
Paso 3: Guardado capa Bronze   

In [0]:
from pyspark.sql.functions import col, count, min, max, avg, when, year, month
from pyspark.sql.functions import round as spark_round
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import to_date




## 1. Carga y tipos
Se cargan los datos de la tabla crada, seleccionan las 16 primeras columnas (excluyendo metadatos) del dataset raw y se cambian `price` a `DoubleType` y `date_of_transfer` a `DateType`.

In [0]:
print("="*70)
print("CARGA DE DATOS")
print("="*70)

df = spark.table("workspace.uk_housing.price_paid")
df = df.select(*df.columns[:16])
display(df.limit(5))

In [0]:
display(df.describe())

In [0]:
df.schema

In [0]:
# ============================================
#  Tipos
# ============================================

df = df.withColumn("price", col("price").cast(DoubleType())) \
       .withColumn("date_of_transfer", to_date(col("date_of_transfer")))

print(f"✓ Registros: {df.count():,}")
print(f"✓ Columnas: {len(df.columns)}")

display(df.limit(5))

## 2. EDA básico
Estadísticas descriptivas de `price` y distribución por tipo de propiedad y año de transferencia.

In [0]:
# ============================================
# EDA básico
# ============================================

display(df.select("price").describe())

display(df.select(
    min("price").alias("min"),
    max("price").alias("max"),
    avg("price").alias("avg")
))

In [0]:
# ============================================
# Tipos de propiedad
# ============================================

display(df.groupBy("property_type").count().orderBy("count", ascending=False))

In [0]:
# ============================================
# Año
# ============================================

df = df.withColumn("year", year("date_of_transfer"))

display(df.groupBy("year").count().orderBy("year"))

## 3. Capa Bronze
Guardado como tabla Delta en `workspace.uk_housing.bronze_property_sales`. Datos sin transformar, solo con tipos corregidos.

In [0]:
# ============================================
# Guardar Bronze 
# ============================================

df.write.format("delta").mode("overwrite").saveAsTable("workspace.uk_housing.bronze_property_sales")

print("Tabla Bronze creada")

display(spark.sql("SELECT COUNT(*) FROM workspace.uk_housing.bronze_property_sales"))